<a href="https://colab.research.google.com/github/lbenit/Floristic_map_africa/blob/main/01_Prep_SEOSAW_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [9]:
# Connect to drive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [13]:
# Print working directory
import os
print(os.getcwd())

/content


# Load in SEOSAW data

In [17]:
# Load in data
stems=pd.read_csv('/content/drive/MyDrive/single_census_stems.csv')
plots=pd.read_csv('/content/drive/MyDrive/single_census_plot.csv')

/tmp/ipykernel_729/1861616270.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  stems=pd.read_csv('/content/drive/MyDrive/single_census_stems.csv')


In [18]:
plots.shape

(404, 219)

# Filter plot data

In [19]:
# Remove pltos with minimum DBH greater than 10
plots_f1 = plots[plots["min_diam_thresh"].le(10) & plots["min_diam_thresh"].notna()]
plots_f1.shape

(404, 219)

In [20]:
# Remove plots outside of study area
remove_list = ["KEN", "COG", "UGA"]   # remove Kenya, Uganda, and Congo

plots_f2 = plots_f1[~plots_f1["country_iso3"].isin(remove_list)]
plots_f2.shape

(404, 219)

In [21]:
# Make sure that plots are after 2012 and has stems
plots_f3 = plots_f2[
    (plots_f2["census_date"] >= 2012) &
    (plots_f2["n_stems"] > 0)
]
plots_f3.shape

(381, 219)

In [22]:
# Sum the plot areas
total_area = plots_f3["plot_area"].sum()
total_area

np.float64(48.27842510404124)

# Filter stem data

In [23]:
# Remove data from filtered plots
stems_f1 = stems[stems["plot_id"].isin(plots_f3["plot_id"])]
stems_f1.shape

(31531, 62)

In [24]:
# Filter stem measurements for everything is above 10cm
stems_f2 = stems_f1[stems_f1["diam"].ge(10)]
stems_f2.shape


(11170, 62)

In [25]:
remove = ['Pinus', 'Eucalyptus', 'Elaeis', 'Lantana',
          'Jacaranda', 'Agave', 'Tectona', 'Mangifera']

# Identify plot_ids that contain ANY removed genus
bad_plots = stems_f2.loc[stems_f2['genus_clean'].isin(remove), 'plot_id'].unique()

# Keep only plot_ids that are NOT in bad_plots
stems_f3 = stems_f2[~stems_f2['plot_id'].isin(bad_plots)]
stems_f3.shape

(11086, 62)

In [26]:
# Filter stems in this list that are herbaceous
herbs=['Pseudognaphalium', 'Sphenoclea']

herb_stems=stems_f3[stems_f3["genus_clean"].isin(herbs)]
herb_stems


,plot_id,subplot_id,tree_id,stem_id,tag_id,census_date,species_orig_binom,species_orig_local,notes_stem,diam,...,ba,diam_adj,agb,wood_density_mean,wood_density_sd,wood_density_lev,wood_density_n,diam_inc,census_int,diam_inc_annual


In [27]:
# Remove those stems from the dataset
stems_f4 = stems_f3[~stems_f3['plot_id'].isin(herb_stems)]
stems_f4.shape

(11086, 62)

In [28]:
# Fix genus name, replace Faragopsis with Fagaropsis
stems_f5 = stems_f4.replace(to_replace='Faragopsis', value='Fagaropsis')

In [29]:
# Show stems call fagaropsis
stems_f5[stems_f5["genus_clean"]=="Fagaropsis"]

,plot_id,subplot_id,tree_id,stem_id,tag_id,census_date,species_orig_binom,species_orig_local,notes_stem,diam,...,ba,diam_adj,agb,wood_density_mean,wood_density_sd,wood_density_lev,wood_density_n,diam_inc,census_int,diam_inc_annual


# Clean up dfs

In [30]:
total_area = plots["plot_area"].sum()
total_area


np.float64(51.168690345343855)

In [31]:
plots.shape

(404, 219)

In [32]:
# Table plots by country
plots_by_country = plots.groupby("country_iso3")["plot_id"].nunique()
plots_by_country

,plot_id
country_iso3,
MOZ,404


In [33]:
# Filter plots data to match stems
final_plots=plots_f3[plots_f3["plot_id"].isin(stems_f5["plot_id"])]
final_plots.shape

(366, 219)

In [34]:
# Select data to keep
cols_to_keep = ["plot_id", "country_iso3", 'plot_shape','plot_area','latitude_of_centre','longitude_of_centre','census_date']

final_plots=final_plots[cols_to_keep]
final_plots.head()


,plot_id,country_iso3,plot_shape,plot_area,latitude_of_centre,longitude_of_centre,census_date
0,MAR_1,MOZ,circle,0.125664,-15.271106,36.485275,2015
1,MAR_2,MOZ,circle,0.125664,-15.269273,36.485329,2015
2,MAR_3,MOZ,circle,0.125664,-15.282653,36.488617,2015
3,MAR_4,MOZ,circle,0.125664,-15.280858,36.488645,2015
4,MAR_5,MOZ,circle,0.125664,-15.272123,36.501926,2015


# Process stem data into abudnances

In [35]:
# See how many genera are only in one plot
# Count how many distinct plots each genus appears in
genus_plot_counts = (
    stems_f3
    .groupby("genus_clean")["plot_id"]
    .nunique()
)

# Filter genera that occur in exactly one plot
genera_in_one_plot = genus_plot_counts[genus_plot_counts == 1]

# Number of such genera
num_genera_in_one_plot = len(genera_in_one_plot)

genera_in_one_plot, num_genera_in_one_plot


(genus_clean
 Adenium           1
 Allophylus        1
 Brackenridgea     1
 Capparis          1
 Dombeya           1
 Ehretia           1
 Gymnanthemum      1
 Hibiscus          1
 Hugonia           1
 Hymenodictyon     1
 Kirkia            1
 Markhamia         1
 Millettia         1
 Ormocarpum        1
 Schinziophyton    1
 Trichilia         1
 Vangueria         1
 Vepris            1
 Ximenia           1
 Name: plot_id, dtype: int64,
 19)

In [36]:
stems=stems_f5

In [37]:
# Calculate basal area
stems["basal_area"] = (stems["diam_adj"]/200) ** 2 * np.pi

In [38]:
# Check ba against basal_area
stems[["basal_area", "ba"]].head()


,basal_area,ba
4,0.038349,0.038708
6,0.007772,0.007854
19,0.010474,0.010568
26,0.030163,0.030481
32,0.048684,0.049087


In [39]:
# Compute the total basal area per plot
basal_area_per_plot = (
    stems
    .groupby("plot_id")["basal_area"]
    .sum()
    .rename('total_basal_area')
    .reset_index()
)


In [40]:
# Marge with stems full df
stems = stems.merge(basal_area_per_plot, on="plot_id", how="left")


In [41]:
# Get proportion basal area
stems["rel_basal_area"] = (stems["basal_area"] / stems["total_basal_area"])


In [42]:
# Summarize relative abundance per genus per plot
rel_abundance = (
    stems
    .groupby(["plot_id", "genus_clean"])["rel_basal_area"]
    .sum()
    .reset_index()
)


In [43]:
# Pivot wider
rel_abundance_wide = (
    rel_abundance
    .pivot(index="plot_id", columns="genus_clean", values="rel_basal_area")
    .fillna(0)
)


In [44]:
# Get length
rel_abundance_wide.shape

(366, 107)

In [45]:
# Check mean indeterminate genus proportion
# This does not run with the test data
rel_abundance_wide["Indet"].mean(),rel_abundance_wide['Theaceae'].mean(),rel_abundance_wide['Caesalpinioideae'].mean(),rel_abundance_wide['Cercidoideae'].mean()

KeyError: 'Theaceae'

In [46]:
# How many plots have Indet >0
(rel_abundance_wide["Indet"] > 0).sum()

np.int64(116)

In [47]:
# Filter out plots with zero indet
rel_abundance_indet = rel_abundance_wide[rel_abundance_wide["Indet"] > 0]
rel_abundance_indet.shape,rel_abundance_indet['Indet'].median()

((116, 107), 0.10064279986375643)

In [48]:
# Remove indeterminate genera or family listed instead of genus
# Filter them out
# This does not work with the subsample of data
rel_abundance_wide = rel_abundance_wide.drop(columns=['Indet', 'Theaceae','Caesalpinioideae','Cercidoideae'])

KeyError: "['Theaceae', 'Caesalpinioideae'] not found in axis"

In [49]:
rel_abundance_wide

genus_clean,Adansonia,Adenium,Afzelia,Aganope,Albizia,Allophylus,Aloe,Androstachys,Annona,Antidesma,...,Trichilia,Uapaca,Vachellia,Vangueria,Vepris,Vernonia,Vitex,Ximenia,Zanha,Ziziphus
plot_id,,,,,,,,,,,,,,,,,,,,,
MAR_1,0.000000,0.0,0.0,0.0,0.152066,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
MAR_10,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
MAR_100,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
MAR_102,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
MAR_105,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.420727,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MAR_92,0.045546,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.026517,0.0
MAR_93,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0
MAR_94,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0


In [50]:
# CHeck that no rows are equal to zero
(rel_abundance_wide.sum(axis=1) > 0).all()

np.True_

In [51]:
# Check that no rows are all zero
row_sums = rel_abundance_wide.sum(axis=1)
bad_rows = row_sums[row_sums <= 0]
bad_rows

,0
plot_id,


In [53]:
# Drop bad rows
rel_abundance_wide = rel_abundance_wide[rel_abundance_wide.sum(axis=1) > 0]


In [54]:
rel_abundance_wide.shape

(366, 107)

In [ ]:
# Save relative abundance df
rel_abundance_wide.to_csv('relative_abundance_data.csv')

# Remove singleton genera

In [56]:
# Drop genera only in one plot and indet
# Combine the genera you want to remove
remove_list = list(genera_in_one_plot.index)

# Filter them out
rel_abundance_wide_remove = rel_abundance_wide.drop(columns=remove_list)



In [57]:
remove_list

['Adenium',
 'Allophylus',
 'Brackenridgea',
 'Capparis',
 'Dombeya',
 'Ehretia',
 'Gymnanthemum',
 'Hibiscus',
 'Hugonia',
 'Hymenodictyon',
 'Kirkia',
 'Markhamia',
 'Millettia',
 'Ormocarpum',
 'Schinziophyton',
 'Trichilia',
 'Vangueria',
 'Vepris',
 'Ximenia']

In [58]:
# CHeck that no rows are equal to zero
(rel_abundance_wide_remove.sum(axis=1) > 0).all()


np.True_

In [59]:
# Check for rows that are all zero
row_sums = rel_abundance_wide_remove.sum(axis=1)
bad_rows = row_sums[row_sums <= 0]
bad_rows


,0
plot_id,


In [60]:
# Remove bad rows
rel_abundance_wide_remove = rel_abundance_wide_remove[rel_abundance_wide_remove.sum(axis=1) > 0]


In [61]:
rel_abundance_wide_remove.shape

(366, 88)

In [ ]:
# Save relative abundance df
rel_abundance_wide_remove.to_csv('relative_abundance_data_no_singelton.csv')

# Filter plot and stem data to match final abundance matrix

In [62]:
# Filter plot data to match plot_id in rel_abundance_wide
final_plots = final_plots[final_plots["plot_id"].isin(rel_abundance_wide.index)]
final_plots.shape

(366, 7)

In [63]:
# Select data to keep
cols_to_keep = ["plot_id", "country_iso3", 'plot_shape','plot_area','latitude_of_centre','longitude_of_centre','census_date']

final_plots=final_plots[cols_to_keep]
final_plots.head()


,plot_id,country_iso3,plot_shape,plot_area,latitude_of_centre,longitude_of_centre,census_date
0,MAR_1,MOZ,circle,0.125664,-15.271106,36.485275,2015
1,MAR_2,MOZ,circle,0.125664,-15.269273,36.485329,2015
2,MAR_3,MOZ,circle,0.125664,-15.282653,36.488617,2015
3,MAR_4,MOZ,circle,0.125664,-15.280858,36.488645,2015
4,MAR_5,MOZ,circle,0.125664,-15.272123,36.501926,2015


In [ ]:
# Save to drive
final_plots.to_csv('selected_seosaw_plots.csv', index=False)

In [64]:
# Filter stem data to match plot_id in rel_abundance_wide
stems_final=stems_f3[stems_f3["plot_id"].isin(rel_abundance_wide.index)]
stems_final.shape

(11086, 62)

In [ ]:
# Save stems to drive
stems_final.to_csv('selected_seosaw_stems.csv', index=False)
